# Version 1 — Prompt Anatomy: an Agent with a Brain, but No Hands

We build an AI Data Analyst agent for an e-commerce marketplace. It can read a question and propose SQL. It cannot run anything.

## Workflow

```text
User Question
     |
     v
AI Data Analyst Agent (model + system prompt)
     |
     v
Generate proposed SQL
     |
     v
Return SQL + explanation + missing information
```

**Tools:** none (`tools=[]`). No database, no retrieval, no validation, no web search.

> “At this stage, our agent has a brain, but no hands. It can understand a question and generate SQL, but it has no tools to access the database.”

## Where this series is going

Each version adds one capability to the same agent.

| Version | Adds | The agent can now… |
|---|---|---|
| **1** | System prompt, no tools | propose SQL |
| 2 | `run_query` | run SQL and read real results |
| 3 | `lookup_schema`, `lookup_examples`, `lookup_common` | look up schema, examples and definitions before writing SQL |
| 4 | `validate_query` | check SQL before running it |
| 5 | `web_search` | use public web sources when the question needs them |
| 6 | Budget middleware | run at most 3 SQL queries per question |
| 7 | Tool-output middleware | get clean, size-limited SQL output |
| 8 | Structured output | return a fixed contract to a supervisor |

## 1. What is prompt anatomy?

A prompt is the set of instructions and information we give the model. A well-structured prompt usually has these parts:

| Component | What it does | Example for our SQL agent |
|---|---|---|
| **Role** | Who the model acts as | You are an AI Data Analyst for an e-commerce marketplace. |
| **Context** | Useful background | The marketplace stores orders, products, customers, payments, and inventory. |
| **Task** | What it must do | Convert a business question into SQL. |
| **Input** | The specific request | How many orders were cancelled? |
| **Constraints** | Rules and limits | Do not invent column names. Do not execute SQL. |
| **Output format** | Shape of the answer | Return SQL and a short explanation. |
| **Examples** | The expected pattern | A sample question and its SQL. |
| **Evaluation criteria** | What a good answer satisfies | SQL addresses the question and uses known schema only. |

Not every prompt needs every part. Pick the parts the task needs.

The system prompt gives the agent its role, context, task, constraints and output format. The model is its **brain**. Tools are its **hands**. In this version there are no tools.

## 2. Our Text-to-SQL prompt

We use these parts:

1. **Role:** AI Data Analyst Agent.
2. **Context:** the marketplace database and its known table names.
3. **Task:** translate a business question into SQL.
4. **Constraints:** read-only SQL only; do not execute; do not invent schema.
5. **Output format:** question, SQL, explanation, assumptions.
6. **Fallback:** if columns or relationships are unknown, say what is missing.

**Limitation:** a list of table names is not a schema. Without column names and relationships, the model cannot promise SQL that runs. The prompt says this explicitly.

The table names are written into the prompt only because this agent has no way to look them up yet. From Version 3 the agent looks them up instead.

## 3. Install dependencies

In [ ]:
%pip install -q langchain langchain-openai python-dotenv

## 4. Configure the model

Set `OPENAI_API_KEY` in your environment or in a `.env` file next to this notebook. The model ID is `gpt-6-astra`; availability depends on your account.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Configuration — Version 1
#
#   .env ──► load_dotenv ──► check OPENAI_API_KEY ──► model (the brain)
#
# Does NOT connect to any database. This version has no tools.
# ─────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY in your environment or .env file.")

MODEL = "gpt-6-astra"
model = ChatOpenAI(model=MODEL)

## 5. Build the system prompt from its parts

Each part has its own heading, so we can inspect and improve one part at a time.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace.

CONTEXT
The database is described by these known table names:
orders, order_items, products, customers, shipments, payment_transactions,
reviews, suppliers, warehouses, inventory, promotions, categories,
customer_addresses, product_suppliers
Only table names are currently known. Column names, data types, keys, and
relationships have not been provided in this version.

TASK
Translate the user's natural-language business question into a proposed SQL query.
Identify the metric, filters, joins, aggregation, and date boundaries required.

CAPABILITIES
You have no tools in this version. You cannot inspect schema metadata, execute SQL,
validate a query against PostgreSQL, or retrieve real database results.

CONSTRAINTS
- Propose read-only SELECT SQL only.
- Never claim that SQL was executed or that results were retrieved.
- Never invent column names, relationships, values, or results.
- If the available schema is insufficient, state what information is missing instead
  of guessing.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, identify the intended numerator and denominator.
- Do not expose private chain-of-thought; provide a concise explanation only.

OUTPUT FORMAT
Return these sections:
QUESTION: Restate the business question briefly.
PROPOSED SQL: Provide SQL only if it can be drafted responsibly; otherwise write UNKNOWN.
EXPLANATION: Briefly explain the query's intent.
ASSUMPTIONS / MISSING INFORMATION: List assumptions and schema details still needed.
"""

## 6. Create the agent — brain, no hands

- `model=model` gives the agent its brain.
- `tools=[]` means no hands: it cannot touch a database.
- `system_prompt=SYSTEM_PROMPT` gives it its instructions.

We use `create_agent()` and do not call the model directly with `llm.invoke()`. Later versions only change the tools, prompt and middleware passed here.

In [ ]:
# ─────────────────────────────────────────────────────────────
# agent = model + prompt + no tools
#
#   question ──► agent.invoke ──► model writes text ──► last message = answer
#
# Does NOT execute SQL. Any SQL in the answer is a proposal only.
# ─────────────────────────────────────────────────────────────
agent = create_agent(
    model=model,
    tools=[],
    system_prompt=SYSTEM_PROMPT,
)


def ask_agent(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content


question = "How many orders were cancelled?"
print("QUESTION:\n" + question)
print("\nAGENT RESPONSE:")
print(ask_agent(question))

## 7. Try more business questions

Watch the **ASSUMPTIONS / MISSING INFORMATION** section. Without columns, the agent should say what it does not know.

In [ ]:
questions = [
    "How many orders were cancelled?",
    "What are the top 10 best-selling products?",
    "What percentage of orders were cancelled?",
    "Which products have inventory below 100 units?",
    "What is the average order value?",
]

for question in questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("=" * 80)
    print(ask_agent(question))

## 8. Key points

- **Prompt anatomy** gives the agent a clear role, context, task, constraints and output format.
- `create_agent()` builds the agent loop; we do not call the model directly.
- The model is the **brain**. Tools are the **hands**. With `tools=[]` the agent cannot run SQL.
- Table names alone are not enough for reliable SQL.
- The agent must not claim it queried the database.

**Next — Version 2:** we give the agent hands with a `run_query` tool. It will run SQL, read the results and answer from real data.